# Imports and Initial Configuration

In [ ]:
import json
import time
import numpy as np
from numpy.linalg import norm
from pydantic import BaseModel, Field
from ollama import chat, embeddings
from pathlib import Path
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder
import torch

# ----- GLOBAL VARIABLES -----
rag_mode = "cross_encoder" # "hybrid", "cross_encoder", "BM25", "Vector"
enable_llm = True # Flag to enable the LLM router execution. If False, only tests the RAG system
model = "llama3:latest" # LLM model used for the router
ratio_queries = 0.1 # Ratio of the dataset to be processed (e.g., 1 = 100%, 0.1 = 10%)
queries_type = "tool_only" # Options: tool_param_all , tool_only , form_queriess
top_k = 3 # Number of top tools to retrieve

# ----- FILE PATHS -----
script_folder = Path().absolute()
tools_list_path = script_folder.parent / 'input' / 'documentation' / 'tools_list_exemples.json'
queries_list_path = script_folder.parent / 'input' / 'tool' / f'{queries_type}.json'
results_path = script_folder / 'output' / f'{rag_mode}_{queries_type}_topk{top_k}.json'

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# Utility Functions and Caching

In [ ]:
def get_vector(text): 
    response = embeddings(model="mxbai-embed-large", prompt=text)
    return response['embedding']

def format_tool_text(tool):
    name = tool.get('name', '')
    desc = tool.get('description', str(tool))
    tags = ", ".join(tool.get('tags', []))
    examples = " ".join(tool.get('examples', []))
    return f"{name} : {desc}. Tags {tags}. Examples {examples}."

def precompute_tool_embeddings(tools_list):
    print("Caching vector embeddings...")
    vecs = []
    for tool in tools_list:
        formatted_text = format_tool_text(tool)
        vector = get_vector(formatted_text)
        vecs.append(vector)
    return np.array(vecs)

def precompute_bm25_cache(tools_list):
    print("Caching BM25 index...")
    tokenized_corpus = []
    for tool in tools_list:
        formatted_text = format_tool_text(tool)
        tokens = formatted_text.lower().split()
        tokenized_corpus.append(tokens)
    return BM25Okapi(tokenized_corpus)

# RAG Retrieval Logic

In [ ]:
def retrieve_tools(user_prompt, tools_list, top_k, mode, tool_matrix=None, bm25_index=None):

    if mode == "vector":
        # 1. FULL VECTOR (Fast, cosine similarity math calculation)
        query_vec = np.array(get_vector(user_prompt))
        cosine_scores = np.dot(tool_matrix, query_vec) / (norm(tool_matrix, axis=1) * norm(query_vec))
        top_indices = np.argsort(cosine_scores)[::-1][:top_k]
        
        results = []
        for i in top_indices:
            results.append({"score": float(cosine_scores[i]), "tool": tools_list[i]})
        return results
        
    elif mode == "bm25":
        # 2. FULL BM25 (Ultra fast lexical search, no LLM calls required)
        tokenized_query = user_prompt.lower().split(" ")
        lexical_scores = bm25_index.get_scores(tokenized_query)
        top_indices = np.argsort(lexical_scores)[::-1][:top_k]
        
        results = []
        for i in top_indices:
            results.append({"score": float(lexical_scores[i]), "tool": tools_list[i]})
        return results

    elif mode == "hybrid":
        # 3. HYBRID WITH RRF (State-of-the-Art Reciprocal Rank Fusion)
        query_vec = np.array(get_vector(user_prompt))
        vector_scores = np.dot(tool_matrix, query_vec) / (norm(tool_matrix, axis=1) * norm(query_vec))
        
        tokenized_query = user_prompt.lower().split(" ")
        lexical_scores = bm25_index.get_scores(tokenized_query)

        k_rrf = 60
        vec_ranks = len(vector_scores) - np.argsort(np.argsort(vector_scores))
        lex_ranks = len(lexical_scores) - np.argsort(np.argsort(lexical_scores))
        
        rrf_vector = 1.0 / (k_rrf + vec_ranks)
        rrf_lexical = 1.0 / (k_rrf + lex_ranks)
        
        final_scores = rrf_vector + rrf_lexical
        top_indices = np.argsort(final_scores)[::-1][:top_k]
        
        results = []
        for i in top_indices:
            results.append({
                "score": float(final_scores[i]), 
                "tool": tools_list[i], 
                "vec_rank": int(vec_ranks[i]),
                "bm25_rank": int(lex_ranks[i])
            })
        return results
        
    elif mode == "cross_encoder":
        # 4. CROSS ENCODER (Computationally heavy but highly accurate reranking)
        pairs = []
        for tool in tools_list:
            formatted_text = format_tool_text(tool)
            pairs.append([user_prompt, formatted_text])
            
        scores = reranker_model.predict(pairs)
        top_indices = np.argsort(scores)[::-1][:top_k]
        
        results = []
        for i in top_indices:
            results.append({"score": float(scores[i]), "tool": tools_list[i]})
        return results

    else:
        raise ValueError(f"Unrecognized RAG mode: {mode}")

# LLM Agent Router

In [ ]:
class RouteDecision(BaseModel):
    reasoning: str = Field(description="Step-by-step analysis comparing the user's request against the available tools before making a decision.")
    confidence: float = Field(description="Confidence level from 0.0 to 1.0")
    selected_tool: str = Field(description="The exact name of the tool. Return 'none' if no tool matches.")

def agent_router(user_prompt: str, relevant_tools: list, model_name: str) -> RouteDecision:
    tools_formatted = "\n".join([
        f"{t.get('name', 'Unknown')}: {t.get('description', '')} (Tags: {', '.join(t.get('tags', []))})" 
        for t in relevant_tools
    ])
    
    system_prompt = f"""You are a Router Agent expert in medical and dental imaging (CBCT, IOS, MRI).
    Your role is to analyze the user's request and select the most relevant tool from the FILTERED list below.
    If none of these {len(relevant_tools)} tools fit perfectly, return 'none'.

    === FILTERED TOOLS ===
    {tools_formatted}

    === ROUTING GUIDELINES & EXAMPLES ===
    - Pay close attention to subtle differences. For example, if a user specifically asks for "batch processing" or "multiple scans", prioritize tools designed for batching (e.g., batchdentalseg).
    - If a user asks to "segment" or "split" specific teeth, ensure the tool handles instance segmentation (e.g., amasss_cli).
    - If the request is for registration, check if it's CBCT-to-CBCT, MRI-to-CBCT, or intraoral (IOS) and choose the specific tool accordingly.

    Carefully analyze the user's prompt step-by-step in the 'reasoning' field BEFORE selecting the tool. Output strictly matching the JSON schema.
    """
    
    try:
        response = chat(
            model=model_name,
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': user_prompt},
            ],
            format=RouteDecision.model_json_schema(),
            options={"temperature": 0},
        )
        return RouteDecision.model_validate_json(response.message.content)
    except Exception as e:
        return RouteDecision(selected_tool="error", confidence=0.0, reasoning=f"Error: {str(e)}")

# Data Initialization and Caching Setup

In [ ]:
# ----- LOAD FILES -----
with open(tools_list_path, 'r', encoding='utf-8') as f:
    tools_list = json.load(f)

with open(queries_list_path, 'r', encoding='utf-8') as f:
    queries_list = json.load(f)

# ----- CACHE -----
t_start_setup = time.time()
tool_matrix = None
bm25_index = None
reranker_model = None

# Initialize CrossEncoder if required
if rag_mode.lower() == "cross_encoder":
    print("Loading CrossEncoder model (BAAI/bge-reranker-v2-m3)...")
    reranker_model = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512, device="cuda")

# Precompute vector embeddings if utilizing vector or hybrid modes
if rag_mode.lower() in ["vector", "hybrid"]:
    tool_matrix = precompute_tool_embeddings(tools_list)

# Precompute BM25 lexical search index if utilizing bm25 or hybrid modes
if rag_mode.lower() in ["bm25", "hybrid"]:
    bm25_index = precompute_bm25_cache(tools_list)

print(f"\nSetup completed in {time.time() - t_start_setup:.2f}s (Mode: {rag_mode.upper()})")

# Terminal Output Display Functions

In [ ]:
def print_rag_block(index, total_queries, prompt, rag_results, mode, expected_tool, rag_match, rag_perfect_match):
    if rag_perfect_match:
        rag_hit_sign = "✅"
    elif rag_match:
        rag_hit_sign = "☑️"
    else:
        rag_hit_sign = "❌"

    print(f"\n{'='*65}")
    print(f"Prompt ({index}/{total_queries}) : '{prompt}'")
    print(f"{'-'*65}")
    print(f"I: RAG SEARCH ({mode.upper()}) {rag_hit_sign}")
    print(f"{'-'*65}")
    
    for i, res in enumerate(rag_results, start=1):
        tool_name = res["tool"].get('name', 'Unknown')
        
        is_expected = "->" if tool_name == expected_tool else "  "
        
        if mode == "hybrid":
            print(f"{is_expected} {i} Score: {res['score']:.4f} (Vec Rank: {res['vec_rank']}, BM25 Rank: {res['bm25_rank']}) tool: {tool_name}")
        else:
            print(f"{is_expected} {i} Score: {res['score']:.4f} tool: {tool_name}")
    print(f"{'-'*65}")

def print_llm_block(decision, expected_tool, llm_status_icon, latency):
    print("II: LLM ROUTER DECISION")
    print(f"{'-'*65}")  
    print(f"Tool chosen: {decision.selected_tool} {llm_status_icon} ")
    print(f"Expected   : {expected_tool}")
    print(f"Confidence : {decision.confidence * 100:.2f}%")
    print(f"Latency    : {latency:.2f} seconds")
    print(f"Reasoning  : {decision.reasoning}")

# Main Loop

In [ ]:
# ----- METRICS INITIALIZATION -----
rag_correct_count = 0
rag_perfect_count = 0
llm_correct_count = 0
total_rag_latency = 0
total_llm_latency = 0
results_detail = []

# Calculate the total number of queries to process based on the ratio
total_queries = int(len(queries_list) * ratio_queries)
queries_to_run = queries_list[:total_queries]

# Iterate over each query in the dataset
for index, q in enumerate(queries_to_run, start=1):
    # Check if q is a list or dict to be flexible
    if isinstance(q, dict):
        prompt = q['query']
        expected_tool = q['expected_tool']
    else:
        prompt = q[0]
        expected_tool = q[1]
    
    # ----- 1. RAG RETRIEVAL PHASE -----
    t0_rag = time.time()
    rag_results = retrieve_tools(prompt, tools_list, top_k, rag_mode.lower(), tool_matrix, bm25_index)
    t1_rag = time.time()
    rag_latency = t1_rag - t0_rag
    total_rag_latency += rag_latency
    
    # Evaluate RAG success
    retrieved_tools = [res["tool"].get("name") for res in rag_results]
    rag_match = expected_tool in retrieved_tools
    rag_perfect_match = (expected_tool == retrieved_tools[0]) if retrieved_tools else False
    
    if rag_match:
        rag_correct_count += 1
    if rag_perfect_match:
        rag_perfect_count += 1
        
    # Print RAG evaluation block
    print_rag_block(index, total_queries, prompt, rag_results, rag_mode, expected_tool, rag_match, rag_perfect_match)
    
    # ----- 2. LLM ROUTER PHASE -----
    llm_latency = 0
    llm_match = False
    decision = None
    
    if enable_llm:
        t0_llm = time.time()
        decision = agent_router(prompt, [res["tool"] for res in rag_results], model)
        t1_llm = time.time()
        llm_latency = t1_llm - t0_llm
        total_llm_latency += llm_latency
        
        # Evaluate LLM success
        llm_match = (decision.selected_tool == expected_tool)
        if llm_match:
            llm_correct_count += 1
            
        llm_status_icon = "✅" if llm_match else "❌"
        # Print LLM decision block
        print_llm_block(decision, expected_tool, llm_status_icon, llm_latency)
        
    # Store detailed results for the current query
    results_detail.append({
        "prompt": prompt,
        "expected_tool": expected_tool,
        "rag_match": rag_match,
        "rag_perfect_match": rag_perfect_match,
        "rag_latency": rag_latency,
        "llm_selected_tool": decision.selected_tool if decision else None,
        "llm_match": llm_match,
        "llm_latency": llm_latency,
        "confidence": decision.confidence if decision else None,
        "reasoning": decision.reasoning if decision else None
    })

# Metrics Calculation and Results Export

In [ ]:
# ----- FINAL METRICS CALCULATION -----
# Calculate percentages and averages safely (avoid division by zero)
rag_accuracy = (rag_correct_count / total_queries) * 100 if total_queries > 0 else 0
rag_perfect_accuracy = (rag_perfect_count / total_queries) * 100 if total_queries > 0 else 0
llm_accuracy = (llm_correct_count / total_queries) * 100 if total_queries > 0 else 0
avg_rag_latency = total_rag_latency / total_queries if total_queries > 0 else 0
avg_llm_latency = total_llm_latency / total_queries if total_queries > 0 else 0
avg_total_latency = avg_rag_latency + (avg_llm_latency if enable_llm else 0)

# Build the final summary dictionary
summary = {
    "rag_mode": rag_mode,
    "enable_llm": enable_llm,
    "queries_type": queries_type,
    "top_k": top_k,
    "model": model,
    "metrics": {
        "total_queries": total_queries,
        "rag_correct": rag_correct_count,
        "rag_accuracy": round(rag_accuracy, 2),
        "rag_perfect_correct": rag_perfect_count,
        "rag_perfect_accuracy": round(rag_perfect_accuracy, 2),
        "llm_correct": llm_correct_count,
        "llm_accuracy": round(llm_accuracy, 2),
        "avg_rag_latency": round(avg_rag_latency, 4),
        "avg_llm_latency": round(avg_llm_latency, 4),
        "avg_total_latency": round(avg_total_latency, 4)
    },
    "details": results_detail,
}

# ----- EXPORT RESULTS -----
# Ensure the output directory exists
results_path.parent.mkdir(parents=True, exist_ok=True)

# Save the benchmark results to a JSON file
with open(results_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=4, ensure_ascii=False)

# ----- FINAL SUMMARY OUTPUT TO TERMINAL -----
print("\n === BENCHMARK COMPLETED ===")
print(f"RAG Accuracy (Top {top_k}): {rag_accuracy:.2f}% ({rag_correct_count}/{total_queries})")
print(f"RAG Perfect Accuracy (Top 1): {rag_perfect_accuracy:.2f}% ({rag_perfect_count}/{total_queries})")

if enable_llm:
    print(f"LLM Accuracy (Exact): {llm_accuracy:.2f}% ({llm_correct_count}/{total_queries})")

print(f"Average RAG Latency:  {avg_rag_latency:.4f} sec")

if enable_llm:
    print(f"Average LLM Latency:  {avg_llm_latency:.4f} sec")
    print(f"Average Total Latency:{avg_total_latency:.4f} sec")